# Lab 22 — Attention and Transformers for Time Series with TensorFlow

[Open this lab in Google Colab](https://colab.research.google.com/github/wanghemath/Book-MachineLearning2sda/blob/main/labs/chapter-22-attention-transformers-time-series-lab.ipynb)

This lab is designed for independent study. It develops the mathematical ideas first, then turns them into TensorFlow/Keras code for time-series forecasting.

## Learning goals

By the end of this lab, you should be able to:

1. Explain the query-key-value view of attention.
2. Compute scaled dot-product attention from scratch.
3. Explain why position information is needed in a Transformer.
4. Convert a time series into a supervised window-to-horizon forecasting dataset.
5. Build an encoder-style Transformer forecasting model in TensorFlow/Keras.
6. Compare a Transformer with baseline and MLP forecasting models.
7. Inspect attention-score shapes and interpret attention heatmaps carefully.

We use TensorFlow/Keras because it is available in Google Colab and provides a concise implementation of multi-head attention.


## 1. Mathematical background: attention

A sequence model receives an ordered block of observations

$$
X_{t-L+1:t} = (x_{t-L+1}, x_{t-L+2}, \ldots, x_t),
$$

where each $x_i$ may contain one or more features. A Transformer first maps each observation to a vector representation. If the embedded sequence is

$$
H \in R^{L \times d},
$$

then one self-attention head forms three matrices:

$$
Q = HW_Q, \qquad K = HW_K, \qquad V = HW_V.
$$

For each time position, attention compares a query vector with key vectors from all positions. The scaled dot-product attention matrix is

$$
A = \operatorname{softmax}\left(\frac{QK^T}{\sqrt{d_k}} + M\right),
$$

where $M$ is an optional mask. The output is

$$
Z = AV.
$$

Each row of $A$ is a distribution over input positions. A large entry $A_{ij}$ means that position $i$ uses information from position $j$ strongly.

### Why scale by $\sqrt{d_k}$?

If dot products become very large, the softmax can become nearly one-hot. That can make gradients unstable or uninformative. Dividing by $\sqrt{d_k}$ keeps the score scale more stable as the attention dimension increases.

### Why use several heads?

Multi-head attention repeats this process with several learned projections. Different heads can learn different temporal relationships: short lags, seasonal lags, event effects, or slow trend information.


## 2. Causal versus noncausal attention

For forecasting, we must avoid using future information. There are two related issues:

1. **Dataset leakage:** the training features must not include future target values.
2. **Architectural leakage:** a model should not allow a token representation to look ahead when the task requires autoregressive processing.

In this lab, our input window only contains historical observations, and the model predicts a future horizon. This is already safe at the dataset level. We will also demonstrate a causal mask so that you understand how autoregressive Transformer decoders prevent each position from seeing future positions.


## 3. Programming setup

The next cell imports the packages used in the lab. If you run this outside Colab and TensorFlow is missing, install it first or use Colab.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "TensorFlow is required for this lab. In Google Colab it is usually preinstalled. "
        "If needed, run: pip install tensorflow"
    ) from exc

np.random.seed(7339)
tf.keras.utils.set_random_seed(7339)

print("TensorFlow version:", tf.__version__)


## 4. Attention from scratch with NumPy

Before using Keras, we implement the core operation directly. This helps you understand what `MultiHeadAttention` computes internally.

The input matrices have shapes:

- `Q`: number of query positions by key dimension.
- `K`: number of key positions by key dimension.
- `V`: number of key positions by value dimension.

The output has one vector for each query position.


In [ ]:
def softmax_rows(scores):
    shifted = scores - np.max(scores, axis=-1, keepdims=True)
    exp_scores = np.exp(shifted)
    return exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)


def scaled_dot_product_attention(Q, K, V, mask=None):
    key_dim = Q.shape[-1]
    scores = Q @ K.T / np.sqrt(key_dim)
    if mask is not None:
        scores = scores + mask
    weights = softmax_rows(scores)
    output = weights @ V
    return output, weights, scores

# A tiny sequence with four time positions and three-dimensional embeddings.
rng = np.random.default_rng(7339)
H_toy = rng.normal(size=(4, 3))
Wq = rng.normal(size=(3, 2))
Wk = rng.normal(size=(3, 2))
Wv = rng.normal(size=(3, 2))

Q = H_toy @ Wq
K = H_toy @ Wk
V = H_toy @ Wv

Z, A, scores = scaled_dot_product_attention(Q, K, V)

print("Attention weights:")
print(np.round(A, 3))
print("Row sums:", np.round(A.sum(axis=1), 6))
print("Output shape:", Z.shape)


### Interpretation checkpoint

The attention matrix is not a covariance matrix and not a correlation matrix. Its rows are probability distributions over positions after the softmax. The matrix is usually not symmetric.


In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(A, aspect="auto")
plt.colorbar(label="attention weight")
plt.xlabel("key/value position")
plt.ylabel("query position")
plt.title("Toy self-attention weights")
plt.show()


## 5. Causal mask from scratch

A causal mask sets future scores to a very negative number before the softmax. For a sequence of length $L$, query position $i$ cannot attend to key position $j$ when $j > i$.


In [ ]:
L = 5
causal_mask = np.triu(np.ones((L, L)), k=1)
causal_mask = np.where(causal_mask == 1, -1e9, 0.0)

H_small = rng.normal(size=(L, 4))
Q_small = H_small[:, :2]
K_small = H_small[:, 2:4]
V_small = H_small[:, :2]

_, A_causal, _ = scaled_dot_product_attention(Q_small, K_small, V_small, mask=causal_mask)

print(np.round(A_causal, 3))
print("Weights above the diagonal should be approximately zero.")


In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(A_causal, aspect="auto")
plt.colorbar(label="attention weight")
plt.xlabel("key/value position")
plt.ylabel("query position")
plt.title("Causal self-attention weights")
plt.show()


## 6. Simulate a time series with local and long-range effects

A Transformer is useful when the model may need to combine information from several parts of a window. We simulate a series with:

- a slow trend,
- daily seasonality,
- weekly seasonality,
- random event pulses whose effects last for many time steps,
- autoregressive noise.

The task is to predict the next 24 observations from the previous 96 observations.


In [ ]:
def simulate_transformer_series(n=2200, seed=7339):
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    daily = np.sin(2 * np.pi * t / 24)
    weekly = np.cos(2 * np.pi * t / (24 * 7))
    slow_trend = 0.0006 * t

    event = np.zeros(n)
    event_times = rng.choice(np.arange(80, n - 120), size=24, replace=False)
    event[event_times] = 1.0

    event_memory = np.zeros(n)
    for e in event_times:
        length = min(96, n - e)
        event_memory[e:e + length] += np.exp(-np.arange(length) / 30)

    ar_noise = np.zeros(n)
    innovations = rng.normal(scale=0.35, size=n)
    for i in range(1, n):
        ar_noise[i] = 0.65 * ar_noise[i - 1] + innovations[i]

    y = 8 + slow_trend + 1.4 * daily + 0.9 * weekly + 1.8 * event_memory + ar_noise

    df = pd.DataFrame({
        "time": t,
        "y": y,
        "daily_sin": np.sin(2 * np.pi * t / 24),
        "daily_cos": np.cos(2 * np.pi * t / 24),
        "weekly_sin": np.sin(2 * np.pi * t / (24 * 7)),
        "weekly_cos": np.cos(2 * np.pi * t / (24 * 7)),
        "event": event,
        "event_memory_true": event_memory,
    })
    return df


df = simulate_transformer_series()
df.head()


In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(df["time"], df["y"], linewidth=1)
plt.scatter(df.loc[df["event"] == 1, "time"], df.loc[df["event"] == 1, "y"], s=25, label="event")
plt.xlabel("time")
plt.ylabel("y")
plt.title("Synthetic time series with seasonal, local, and long-range event effects")
plt.legend()
plt.show()


## 7. Turn the series into supervised learning windows

For each training example, the input is a matrix with shape

$$
\text{input width} \times \text{number of features}.
$$

The target is a vector containing the next `horizon` values of the response.

Important: the split must be chronological. Random splitting would leak future regimes into the training data.


In [ ]:
INPUT_WIDTH = 96
HORIZON = 24
FEATURE_COLS = ["y", "daily_sin", "daily_cos", "weekly_sin", "weekly_cos", "event"]
TARGET_COL = "y"
TARGET_INDEX_IN_FEATURES = FEATURE_COLS.index(TARGET_COL)


def make_windows(df, feature_cols, target_col, input_width, horizon):
    features = df[feature_cols].to_numpy(dtype=np.float32)
    target = df[target_col].to_numpy(dtype=np.float32)
    X, Y, end_times = [], [], []
    last_start = len(df) - input_width - horizon + 1
    for start in range(last_start):
        end = start + input_width
        X.append(features[start:end])
        Y.append(target[end:end + horizon])
        end_times.append(end)
    return np.asarray(X), np.asarray(Y), np.asarray(end_times)


X_raw, y_raw, end_times = make_windows(df, FEATURE_COLS, TARGET_COL, INPUT_WIDTH, HORIZON)
print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)


In [ ]:
def chronological_split(X, y, times, train_frac=0.70, val_frac=0.15):
    n = len(X)
    n_train = int(train_frac * n)
    n_val = int(val_frac * n)
    X_train = X[:n_train]
    y_train = y[:n_train]
    t_train = times[:n_train]

    X_val = X[n_train:n_train + n_val]
    y_val = y[n_train:n_train + n_val]
    t_val = times[n_train:n_train + n_val]

    X_test = X[n_train + n_val:]
    y_test = y[n_train + n_val:]
    t_test = times[n_train + n_val:]
    return (X_train, y_train, t_train), (X_val, y_val, t_val), (X_test, y_test, t_test)


(train_raw, val_raw, test_raw) = chronological_split(X_raw, y_raw, end_times)
X_train_raw, y_train_raw, t_train = train_raw
X_val_raw, y_val_raw, t_val = val_raw
X_test_raw, y_test_raw, t_test = test_raw

print("Train:", X_train_raw.shape, y_train_raw.shape)
print("Validation:", X_val_raw.shape, y_val_raw.shape)
print("Test:", X_test_raw.shape, y_test_raw.shape)


## 8. Scale using training data only

Scaling parameters must be computed only from the training set. Otherwise information from the validation or test periods leaks into the model.


In [ ]:
feature_mean = X_train_raw.mean(axis=(0, 1), keepdims=True)
feature_std = X_train_raw.std(axis=(0, 1), keepdims=True)
feature_std = np.where(feature_std < 1e-6, 1.0, feature_std)

target_mean = y_train_raw.mean()
target_std = y_train_raw.std()

X_train = (X_train_raw - feature_mean) / feature_std
X_val = (X_val_raw - feature_mean) / feature_std
X_test = (X_test_raw - feature_mean) / feature_std

y_train = (y_train_raw - target_mean) / target_std
y_val = (y_val_raw - target_mean) / target_std
y_test = (y_test_raw - target_mean) / target_std

print("Feature mean shape:", feature_mean.shape)
print("Scaled training target mean:", round(float(y_train.mean()), 4))


## 9. Baseline forecasts

Before fitting a neural network, always build baselines. A sophisticated model is useful only if it improves on simple alternatives.

We use two baselines:

1. **Persistence:** every future value equals the last observed value.
2. **Daily seasonal naive:** the next 24 values repeat the last observed 24-hour pattern.


In [ ]:
def inverse_target(z):
    return z * target_std + target_mean


def persistence_forecast(X_raw_window, horizon, target_index):
    last_value = X_raw_window[:, -1, target_index]
    return np.repeat(last_value[:, None], horizon, axis=1)


def seasonal_naive_forecast(X_raw_window, horizon, target_index, season_length=24):
    if season_length < horizon:
        raise ValueError("This simple implementation assumes season_length >= horizon.")
    return X_raw_window[:, -season_length:-season_length + horizon, target_index]


def rmse_by_horizon(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2, axis=0))


def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))


persist_test = persistence_forecast(X_test_raw, HORIZON, TARGET_INDEX_IN_FEATURES)
seasonal_test = seasonal_naive_forecast(X_test_raw, HORIZON, TARGET_INDEX_IN_FEATURES)

baseline_table = pd.DataFrame({
    "model": ["persistence", "daily seasonal naive"],
    "test_RMSE": [rmse(y_test_raw, persist_test), rmse(y_test_raw, seasonal_test)],
    "test_MAE": [mae(y_test_raw, persist_test), mae(y_test_raw, seasonal_test)],
})
baseline_table


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, HORIZON + 1), rmse_by_horizon(y_test_raw, persist_test), marker="o", label="persistence")
plt.plot(np.arange(1, HORIZON + 1), rmse_by_horizon(y_test_raw, seasonal_test), marker="o", label="daily seasonal naive")
plt.xlabel("forecast horizon")
plt.ylabel("RMSE")
plt.title("Baseline horizon-wise error")
plt.legend()
plt.show()


## 10. Positional information

Self-attention alone does not know whether an observation came first, last, or somewhere in the middle. We need position information.

Two common choices are:

1. **Learned position embeddings:** the model learns a vector for each position.
2. **Sinusoidal position encodings:** deterministic sine and cosine features at multiple frequencies.

The Keras Transformer model below uses learned position embeddings. The next cell also shows sinusoidal encodings for intuition.


In [ ]:
def sinusoidal_position_encoding(length, d_model):
    position = np.arange(length)[:, None]
    div_term = np.exp(np.arange(0, d_model, 2) * (-np.log(10000.0) / d_model))
    encoding = np.zeros((length, d_model))
    encoding[:, 0::2] = np.sin(position * div_term)
    encoding[:, 1::2] = np.cos(position * div_term[:encoding[:, 1::2].shape[1]])
    return encoding


pe = sinusoidal_position_encoding(INPUT_WIDTH, 16)
plt.figure(figsize=(10, 4))
plt.imshow(pe.T, aspect="auto")
plt.colorbar(label="encoding value")
plt.xlabel("time position in input window")
plt.ylabel("encoding coordinate")
plt.title("Sinusoidal positional encoding")
plt.show()


## 11. TensorFlow data pipelines

For small data, NumPy arrays are enough. For a clean neural-network workflow, we create TensorFlow datasets with shuffling only for the training windows.


In [ ]:
BATCH_SIZE = 64

train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(buffer_size=len(X_train), seed=7339).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

for xb, yb in train_ds.take(1):
    print("Batch input shape:", xb.shape)
    print("Batch target shape:", yb.shape)


## 12. Helper functions for training and evaluation

We train for a small number of epochs so that the lab runs comfortably in Colab. You can increase the number of epochs for a final project.


In [ ]:
def compile_and_fit(model, train_data, val_data, max_epochs=30, learning_rate=1e-3):
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="mse",
        metrics=[keras.metrics.MeanAbsoluteError(name="mae")],
    )
    early_stop = keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
    )
    history = model.fit(
        train_data,
        validation_data=val_data,
        epochs=max_epochs,
        callbacks=[early_stop],
        verbose=1,
    )
    return history


def plot_history(history, title):
    hist = pd.DataFrame(history.history)
    plt.figure(figsize=(7, 4))
    plt.plot(hist["loss"], label="train loss")
    plt.plot(hist["val_loss"], label="validation loss")
    plt.xlabel("epoch")
    plt.ylabel("MSE loss")
    plt.title(title)
    plt.legend()
    plt.show()


def evaluate_model(model, X_scaled, y_true_raw, name):
    pred_scaled = model.predict(X_scaled, verbose=0)
    pred_raw = inverse_target(pred_scaled)
    return {
        "model": name,
        "test_RMSE": rmse(y_true_raw, pred_raw),
        "test_MAE": mae(y_true_raw, pred_raw),
    }, pred_raw


## 13. MLP baseline model

An MLP can use all values in the input window after flattening. It does not have a built-in idea of temporal order, except through the order of flattened features.


In [ ]:
def build_mlp_model(input_width, n_features, horizon):
    inputs = keras.Input(shape=(input_width, n_features))
    x = layers.Flatten()(inputs)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.15)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(horizon)(x)
    return keras.Model(inputs, outputs, name="mlp_forecaster")


mlp_model = build_mlp_model(INPUT_WIDTH, len(FEATURE_COLS), HORIZON)
mlp_model.summary()


In [ ]:
mlp_history = compile_and_fit(mlp_model, train_ds, val_ds, max_epochs=25)
plot_history(mlp_history, "MLP training history")

mlp_metrics, mlp_pred = evaluate_model(mlp_model, X_test, y_test_raw, "MLP")
mlp_metrics


## 14. Transformer encoder block

A Transformer encoder block has two residual sublayers:

1. Multi-head self-attention.
2. A position-wise feedforward network.

Layer normalization and residual connections make the optimization more stable.


In [ ]:
class PositionalEmbedding(layers.Layer):
    def __init__(self, sequence_length, d_model, **kwargs):
        super().__init__(**kwargs)
        self.sequence_length = sequence_length
        self.d_model = d_model
        self.position_embedding = layers.Embedding(
            input_dim=sequence_length,
            output_dim=d_model,
        )

    def call(self, inputs):
        length = tf.shape(inputs)[1]
        positions = tf.range(start=0, limit=length, delta=1)
        embedded_positions = self.position_embedding(positions)
        return inputs + embedded_positions

    def get_config(self):
        config = super().get_config()
        config.update({"sequence_length": self.sequence_length, "d_model": self.d_model})
        return config


def transformer_encoder_block(x, d_model, num_heads, key_dim, ff_dim, dropout_rate, causal=False):
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=key_dim,
        dropout=dropout_rate,
    )(x, x, use_causal_mask=causal)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attention_output)

    feedforward = layers.Dense(ff_dim, activation="relu")(x)
    feedforward = layers.Dropout(dropout_rate)(feedforward)
    feedforward = layers.Dense(d_model)(feedforward)
    x = layers.LayerNormalization(epsilon=1e-6)(x + feedforward)
    return x


## 15. Build an encoder-style Transformer forecaster

The model is not a language model. It is a window-to-vector forecaster:

$$
\text{past window} \longmapsto \text{next horizon}.
$$

After Transformer blocks, we pool over the input window and output a vector of length 24.


In [ ]:
def build_transformer_forecaster(
    input_width,
    n_features,
    horizon,
    d_model=64,
    num_heads=4,
    key_dim=16,
    ff_dim=128,
    num_blocks=2,
    dropout_rate=0.15,
    causal=False,
):
    inputs = keras.Input(shape=(input_width, n_features))
    x = layers.Dense(d_model)(inputs)
    x = PositionalEmbedding(input_width, d_model)(x)

    for _ in range(num_blocks):
        x = transformer_encoder_block(
            x,
            d_model=d_model,
            num_heads=num_heads,
            key_dim=key_dim,
            ff_dim=ff_dim,
            dropout_rate=dropout_rate,
            causal=causal,
        )

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(64, activation="relu")(x)
    outputs = layers.Dense(horizon)(x)
    return keras.Model(inputs, outputs, name="transformer_forecaster")


transformer_model = build_transformer_forecaster(
    INPUT_WIDTH,
    len(FEATURE_COLS),
    HORIZON,
    causal=False,
)
transformer_model.summary()


In [ ]:
transformer_history = compile_and_fit(transformer_model, train_ds, val_ds, max_epochs=30, learning_rate=8e-4)
plot_history(transformer_history, "Transformer training history")

transformer_metrics, transformer_pred = evaluate_model(transformer_model, X_test, y_test_raw, "Transformer")
transformer_metrics


## 16. Compare all models

The table below compares baselines and neural models on the test period.


In [ ]:
results = baseline_table.copy()
results = pd.concat([
    results,
    pd.DataFrame([mlp_metrics, transformer_metrics]),
], ignore_index=True)
results.sort_values("test_RMSE")


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, HORIZON + 1), rmse_by_horizon(y_test_raw, seasonal_test), marker="o", label="seasonal naive")
plt.plot(np.arange(1, HORIZON + 1), rmse_by_horizon(y_test_raw, mlp_pred), marker="o", label="MLP")
plt.plot(np.arange(1, HORIZON + 1), rmse_by_horizon(y_test_raw, transformer_pred), marker="o", label="Transformer")
plt.xlabel("forecast horizon")
plt.ylabel("RMSE")
plt.title("Horizon-wise test error")
plt.legend()
plt.show()


## 17. Visualize forecasts

A good forecasting model should not only minimize one summary metric. It should also behave sensibly across different regimes.


In [ ]:
def plot_forecast_case(case_index):
    history_time = np.arange(INPUT_WIDTH)
    future_time = np.arange(INPUT_WIDTH, INPUT_WIDTH + HORIZON)

    past_y = X_test_raw[case_index, :, TARGET_INDEX_IN_FEATURES]
    true_future = y_test_raw[case_index]

    plt.figure(figsize=(10, 4))
    plt.plot(history_time, past_y, label="history")
    plt.plot(future_time, true_future, marker="o", label="true future")
    plt.plot(future_time, seasonal_test[case_index], marker="o", label="seasonal naive")
    plt.plot(future_time, transformer_pred[case_index], marker="o", label="Transformer")
    plt.axvline(INPUT_WIDTH - 1, linestyle="--")
    plt.xlabel("relative time")
    plt.ylabel("y")
    plt.title(f"Forecast example {case_index}")
    plt.legend()
    plt.show()


plot_forecast_case(0)
plot_forecast_case(20)


## 18. Inspect attention-score shapes

Keras can return attention scores from a `MultiHeadAttention` layer. The next model is a small inspection model. It is separate from the trained forecaster, so use it mainly to understand shapes. In a research project, you can build a custom model that exposes scores from the trained network.

For self-attention with `num_heads` heads and an input window of length $L$, attention scores have shape

$$
\text{batch size} \times \text{num heads} \times L \times L.
$$


In [ ]:
def build_attention_inspection_model(input_width, n_features, d_model=32, num_heads=2, key_dim=16):
    inputs = keras.Input(shape=(input_width, n_features))
    x = layers.Dense(d_model)(inputs)
    x = PositionalEmbedding(input_width, d_model)(x)
    attention_layer = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim)
    attention_output, attention_scores = attention_layer(
        x,
        x,
        return_attention_scores=True,
    )
    return keras.Model(inputs, [attention_output, attention_scores], name="attention_inspection_model")


inspection_model = build_attention_inspection_model(INPUT_WIDTH, len(FEATURE_COLS))
attention_output, attention_scores = inspection_model(X_test[:2], training=False)
print("Attention output shape:", attention_output.shape)
print("Attention scores shape:", attention_scores.shape)


In [ ]:
# Plot one head from one example.
score_matrix = attention_scores[0, 0].numpy()

plt.figure(figsize=(6, 5))
plt.imshow(score_matrix, aspect="auto")
plt.colorbar(label="attention weight")
plt.xlabel("key/value position")
plt.ylabel("query position")
plt.title("Example attention scores from an untrained inspection model")
plt.show()


### Interpretation warning

Attention visualizations can be useful, but they are not automatic explanations. A large attention weight means the model used that value in one learned averaging step. It does not prove causality, scientific importance, or policy relevance.


## 19. Optional experiment: causal Transformer block

The previous forecaster uses a historical input window to predict the future, so it does not include future target values. The following model also applies a causal mask inside self-attention. Compare its performance with the noncausal encoder-style model.


In [ ]:
causal_transformer_model = build_transformer_forecaster(
    INPUT_WIDTH,
    len(FEATURE_COLS),
    HORIZON,
    d_model=64,
    num_heads=4,
    key_dim=16,
    ff_dim=128,
    num_blocks=2,
    dropout_rate=0.15,
    causal=True,
)

causal_history = compile_and_fit(causal_transformer_model, train_ds, val_ds, max_epochs=30, learning_rate=8e-4)
plot_history(causal_history, "Causal Transformer training history")

causal_metrics, causal_pred = evaluate_model(causal_transformer_model, X_test, y_test_raw, "Causal Transformer")
causal_metrics


In [ ]:
results_with_causal = pd.concat([
    results,
    pd.DataFrame([causal_metrics]),
], ignore_index=True)
results_with_causal.sort_values("test_RMSE")


## 20. Exercises

1. Change `INPUT_WIDTH` from 96 to 48. Does the Transformer still improve over the baselines?
2. Remove the `event` feature from `FEATURE_COLS`. How does performance change near event periods?
3. Increase `num_heads` from 4 to 8. Does the model improve, or does it overfit?
4. Replace `GlobalAveragePooling1D` with `layers.GlobalMaxPooling1D`. Compare the test error.
5. Build a model with one Transformer block and another with four blocks. Compare training time and validation loss.
6. Explain the difference between dataset leakage and architectural leakage.
7. Plot forecast examples where the model performs badly. What patterns are hard for the model?


## 21. Mini-project ideas

Choose one of the following and write a short report.

### Project A: Attention for event-driven forecasting

Simulate stronger or weaker event effects. Test whether Transformer performance changes as the event-memory length changes.

### Project B: Transformer versus CNN versus GRU

Use the same train-validation-test split and compare models from Labs 20, 21, and 22.

### Project C: Forecast horizon design

Repeat the lab for horizons 6, 12, 24, and 48. Study when the Transformer becomes more or less useful.

### Project D: Interpretable attention experiment

Train a custom model that returns attention scores from the trained forecaster. Study whether attention heads focus on recent lags, seasonal lags, or event positions.


## 22. AI-assisted study prompts

Use these prompts to guide your independent study. Do not paste code blindly; ask the AI to explain each step.

1. "Explain scaled dot-product attention using a time-series forecasting example."
2. "Why does a Transformer need positional information for time series?"
3. "What is the difference between an encoder-style Transformer forecaster and an autoregressive Transformer decoder?"
4. "Help me check whether my time-series train/test split causes leakage."
5. "Interpret this horizon-wise error plot and suggest one model improvement."
6. "Explain why attention weights are not the same thing as causal importance."

When using AI, always verify the answer with the definitions, code, and diagnostic plots in this notebook.
